In [8]:
import numpy as np
import pandas as pd
import os
import json


In [9]:

os.chdir('/Users/duc.tien/Documents/TienDo/stock_research')


In [4]:
feature_pickup_data = pd.read_csv('sample-data/feature_pickup.csv')
feature_pickup_data.head(5)


,feature,description,frequency,source_file,derive_fomulas
0,symbol,NaN,quarterly,financial_statements,NaN
1,fiscalYear,NaN,quarterly,financial_statements,NaN
2,period,NaN,quarterly,financial_statements,NaN
3,securityexchangename,NaN,quarterly,financial_statements,NaN
4,revenuefromcontractwithcustomerexcludingassess...,NaN,quarterly,financial_statements,NaN


In [5]:
## create a dim date table
# Create a date range for the last 5 years
start_date = pd.Timestamp.now() - pd.DateOffset(years=5)
end_date = pd.Timestamp.now()
date_range = pd.date_range(start=start_date, end=end_date, freq='D')

# Create the dim_date table
dim_date = pd.DataFrame({
    'date': date_range.strftime('%Y-%m-%d'),
    'is_last_month': (date_range.month == 12).astype(int),
    'is_last_quarter': (date_range.quarter == 4).astype(int),  # Assuming Q4 is the last quarter
    'is_last_week': (date_range.isocalendar().week == 52).astype(int)  # Assuming week 52 is the last week
})

dim_date.reset_index(drop=True, inplace=True)


In [6]:
dim_date


,date,is_last_month,is_last_quarter,is_last_week
0,2020-07-12,0,0,0
1,2020-07-13,0,0,0
2,2020-07-14,0,0,0
3,2020-07-15,0,0,0
4,2020-07-16,0,0,0
...,...,...,...,...
1822,2025-07-08,0,0,0
1823,2025-07-09,0,0,0
1824,2025-07-10,0,0,0
1825,2025-07-11,0,0,0


## report financial_statement data

In [10]:
import json

# Load JSON data from file
with open('sample-data/AAPL_as_reported_financial_statements.json', 'r') as file:
    financial_data = json.load(file)


In [11]:
def parse_financial_data(financial_data):
    """
    Parse financial data and convert to DataFrame.
    
    Args:
        financial_data (list): List of financial data dictionaries
        
    Returns:
        pd.DataFrame: Parsed financial data as DataFrame
    """
    # Get financial statement feature keys
    financial_statement_keys = feature_pickup_data[
        feature_pickup_data.source_file == 'financial_statements'
    ]['feature'].tolist()
    
    parsed_records = []
    
    for record in financial_data:
        parsed_record = {}
        
        for feature in financial_statement_keys:
            if feature in ['fiscalYear', 'period', 'symbol']:
                # These fields are at the top level
                parsed_record[feature] = record.get(feature)
            else:
                # These fields are nested in the 'data' dictionary
                parsed_record[feature] = record.get('data', {}).get(feature)
        
        parsed_records.append(parsed_record)
    
    # Create DataFrame from parsed records
    financial_df = pd.DataFrame(parsed_records)
    financial_df['obs_date'] = (financial_df['fiscalYear'].astype(str) + financial_df['period']).astype('period[Q]').dt.to_timestamp(how='end').dt.date
    return financial_df



In [29]:
df_financial_statements = parse_financial_data(financial_data)


In [25]:
import duckdb


In [26]:
con = duckdb.connect()


In [30]:
import pandas as pd
con.register('dim_date', dim_date)
con.register('financial_statements', df_financial_statements)


In [ ]:
df_financial_statements_snapshot_month_end = con.sql("""
select
    dim.date as snapshot_date,
    fs.*
from
    dim_date dim
inner join
    financial_statements fs
on
    dim.date = fs.obs_date
where
    dim.is_last_month = 1
group by
    dim.date
""").fetchdf()


In [34]:
df_financial_statements.shape


(63, 131)

## fundamental data

In [30]:
import json

# Load JSON data from file
with open('sample-data/AAPL_fundamental_data.json', 'r') as file:
    fundamental_data = json.load(file)

fundamental_data


[{'date': '2025-03-29',
  'symbol': 'AAPL',
  'reportedCurrency': 'USD',
  'cik': '0000320193',
  'filingDate': '2025-05-02',
  'acceptedDate': '2025-05-02 06:00:46',
  'fiscalYear': '2025',
  'period': 'Q2',
  'revenue': 95359000000,
  'costOfRevenue': 50492000000,
  'grossProfit': 44867000000,
  'researchAndDevelopmentExpenses': 8550000000,
  'generalAndAdministrativeExpenses': 0,
  'sellingAndMarketingExpenses': 0,
  'sellingGeneralAndAdministrativeExpenses': 6728000000,
  'otherExpenses': 0,
  'operatingExpenses': 15278000000,
  'costAndExpenses': 65770000000,
  'netInterestIncome': 0,
  'interestIncome': 0,
  'interestExpense': 0,
  'depreciationAndAmortization': 2661000000,
  'ebitda': 32250000000,
  'ebit': 29589000000,
  'nonOperatingIncomeExcludingInterest': 0,
  'operatingIncome': 29589000000,
  'totalOtherIncomeExpensesNet': -279000000,
  'incomeBeforeTax': 29310000000,
  'incomeTaxExpense': 4530000000,
  'netIncomeFromContinuingOperations': 24780000000,
  'netIncomeFromDisc

In [31]:
import json

# Load JSON data from file
with open('sample-data/AAPL_financial_growth.json', 'r') as file:
    fundamental_data_growth = json.load(file)

fundamental_data_growth


[{'symbol': 'AAPL',
  'date': '2025-03-29',
  'fiscalYear': '2025',
  'period': 'Q2',
  'reportedCurrency': 'USD',
  'revenueGrowth': -0.23283185840707965,
  'grossProfitGrowth': -0.2300815100815101,
  'ebitgrowth': -0.30918472170339933,
  'operatingIncomeGrowth': -0.30918472170339933,
  'netIncomeGrowth': -0.3179190751445087,
  'epsgrowth': -0.3153526970954358,
  'epsdilutedGrowth': -0.3125,
  'weightedAverageSharesGrowth': -0.005811139363112599,
  'weightedAverageSharesDilutedGrowth': -0.006252580298220596,
  'dividendsPerShareGrowth': -0.01971838467760047,
  'operatingCashFlowGrowth': -0.19986637715049274,
  'receivablesGrowth': -0.1603210467743567,
  'inventoryGrowth': -0.0928953841701635,
  'assetGrowth': -0.03735123588648154,
  'bookValueperShareGrowth': 0.006417653409830609,
  'debtGrowth': 0.014328660420045661,
  'rdexpenseGrowth': 0.034107402031930335,
  'sgaexpensesGrowth': -0.06229965156794425,
  'freeCashFlowGrowth': -0.2264863863678459,
  'tenYRevenueGrowthPerShare': 1.540

In [32]:
def parse_fundamental_data(financial_data):
    """
    Parse financial data and convert to DataFrame.

    Args:
        financial_data (list): List of financial data dictionaries

    Returns:
        pd.DataFrame: Parsed financial data as DataFrame
    """
    # Extract relevant feature keys for 'fundamental_data'
    feature_keys = financial_data[0].keys()

    # Use list comprehension for concise record parsing
    parsed_records = [
        {feature: record.get(feature, None) for feature in feature_keys}
        for record in financial_data
    ]

    financial_df = pd.DataFrame(parsed_records)

    financial_df['obs_date'] = (financial_df['fiscalYear'].astype(str) + financial_df['period']).astype('period[Q]').dt.to_timestamp(how='end').dt.date

    return financial_df


In [33]:
fundamental_data = parse_fundamental_data(fundamental_data)


In [ ]:
# Calculate YoY growth features for all numeric columns, grouped by obs_date (quarterly)
import numpy as np

# Ensure obs_date is datetime
fundamental_data['obs_date'] = pd.to_datetime(fundamental_data['obs_date'])

# Sort by obs_date to ensure correct diff
fundamental_data = fundamental_data.sort_values('obs_date').reset_index(drop=True)

# Identify numeric columns to calculate YoY growth for
numeric_cols = fundamental_data.select_dtypes(include=[np.number]).columns

# Calculate YoY growth for each numeric column (current value / value 4 quarters ago - 1)
for col in numeric_cols:
    fundamental_data[f'{col}_yoy'] = (
        fundamental_data[col] / fundamental_data[col].shift(4) - 1
    )

fundamental_data


/var/folders/x2/ryc0qkjx49j3l781k7c1p_700000gt/T/ipykernel_12262/1902081522.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fundamental_data[f'{col}_yoy'] = (
/var/folders/x2/ryc0qkjx49j3l781k7c1p_700000gt/T/ipykernel_12262/1902081522.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fundamental_data[f'{col}_yoy'] = (
/var/folders/x2/ryc0qkjx49j3l781k7c1p_700000gt/T/ipykernel_12262/1902081522.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

,date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,revenue,costOfRevenue,...,netCashProvidedByFinancingActivities_yoy,effectOfForexChangesOnCash_yoy,netChangeInCash_yoy,cashAtEndOfPeriod_yoy,cashAtBeginningOfPeriod_yoy,operatingCashFlow_yoy,capitalExpenditure_yoy,freeCashFlow_yoy,incomeTaxesPaid_yoy,interestPaid_yoy
0,1985-09-30,AAPL,USD,0000320193,1985-09-30,1985-09-30 00:00:00,1985,Q4,409700000,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1985-12-31,AAPL,USD,0000320193,1985-12-31,1985-12-31 00:00:00,1986,Q1,533900000,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1986-03-31,AAPL,USD,0000320193,1986-03-31,1986-03-31 00:00:00,1986,Q2,408900000,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1986-06-30,AAPL,USD,0000320193,1986-06-30,1986-06-30 00:00:00,1986,Q3,448300000,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1986-09-30,AAPL,USD,0000320193,1986-09-30,1986-09-30 00:00:00,1986,Q4,510800000,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,2024-03-30,AAPL,USD,0000320193,2024-05-03,2024-05-02 18:04:25,2024,Q2,90753000000,48482000000,...,0.183059,NaN,-2.562173,0.250359,0.910167,-0.205532,-0.315501,-0.193028,0.789474,-1.0
155,2024-06-29,AAPL,USD,0000320193,2024-08-02,2024-08-01 18:03:34,2024,Q3,85777000000,46099000000,...,0.497713,NaN,-3.631275,-0.109138,0.250359,0.093935,0.027711,0.099642,1.210254,-1.0
156,2024-09-28,AAPL,USD,0000320193,2024-11-01,2024-11-01 06:01:36,2024,Q4,94930000000,51051000000,...,0.077528,NaN,2.942789,-0.025832,-0.109138,0.241365,0.344429,0.229895,-0.410584,-1.0
157,2024-12-28,AAPL,USD,0000320193,2025-01-31,2025-01-31 06:01:27,2025,Q1,124300000000,66025000000,...,0.287265,NaN,-0.968319,-0.278148,-0.025832,-0.249655,0.229097,-0.280191,1.570779,NaN


In [45]:
# Print all columns that end with '_yoy'˚
print([col for col in fundamental_data.columns if col.endswith('_yoy')])


['revenue_yoy', 'costOfRevenue_yoy', 'grossProfit_yoy', 'researchAndDevelopmentExpenses_yoy', 'generalAndAdministrativeExpenses_yoy', 'sellingAndMarketingExpenses_yoy', 'sellingGeneralAndAdministrativeExpenses_yoy', 'otherExpenses_yoy', 'operatingExpenses_yoy', 'costAndExpenses_yoy', 'netInterestIncome_yoy', 'interestIncome_yoy', 'interestExpense_yoy', 'depreciationAndAmortization_yoy', 'ebitda_yoy', 'ebit_yoy', 'nonOperatingIncomeExcludingInterest_yoy', 'operatingIncome_yoy', 'totalOtherIncomeExpensesNet_yoy', 'incomeBeforeTax_yoy', 'incomeTaxExpense_yoy', 'netIncomeFromContinuingOperations_yoy', 'netIncomeFromDiscontinuedOperations_yoy', 'otherAdjustmentsToNetIncome_yoy', 'netIncome_yoy', 'netIncomeDeductions_yoy', 'bottomLineNetIncome_yoy', 'eps_yoy', 'epsDiluted_yoy', 'weightedAverageShsOut_yoy', 'weightedAverageShsOutDil_yoy', 'cashAndCashEquivalents_yoy', 'shortTermInvestments_yoy', 'cashAndShortTermInvestments_yoy', 'netReceivables_yoy', 'accountsReceivables_yoy', 'otherReceivab

In [42]:
fundamental_data_growth = parse_fundamental_data(fundamental_data_growth)


In [43]:
fundamental_data_growth


,symbol,date,fiscalYear,period,reportedCurrency,revenueGrowth,grossProfitGrowth,ebitgrowth,operatingIncomeGrowth,netIncomeGrowth,...,threeYShareholdersEquityGrowthPerShare,tenYDividendperShareGrowthPerShare,fiveYDividendperShareGrowthPerShare,threeYDividendperShareGrowthPerShare,ebitdaGrowth,growthCapitalExpenditure,tenYBottomLineNetIncomeGrowthPerShare,fiveYBottomLineNetIncomeGrowthPerShare,threeYBottomLineNetIncomeGrowthPerShare,obs_date
0,AAPL,2025-03-29,2025,Q2,USD,-0.232832,-0.230082,-0.309185,-0.309185,-0.317919,...,0.075969,1.117554,0.295149,0.134907,None,None,None,None,None,2025-06-30
1,AAPL,2024-12-28,2025,Q1,USD,0.309386,0.328084,0.447467,0.447467,1.465391,...,0.008683,1.133413,0.275851,0.122972,None,None,None,None,None,2025-03-31
2,AAPL,2024-09-28,2024,Q4,USD,0.106707,0.105877,0.167206,0.167206,-0.312943,...,-0.019076,1.103591,0.294579,0.135642,None,None,None,None,None,2024-12-31
3,AAPL,2024-06-29,2024,Q3,USD,-0.054830,-0.061342,-0.091326,-0.091326,-0.092571,...,0.128862,1.151703,0.283571,0.124736,None,None,None,None,None,2024-09-30
4,AAPL,2024-03-30,2024,Q2,USD,-0.241037,-0.229405,-0.308944,-0.308944,-0.303102,...,0.166326,1.216607,0.307695,0.170447,None,None,None,None,None,2024-06-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,AAPL,1986-09-30,1986,Q4,USD,0.139416,0.139416,-3.492973,-3.492973,0.018576,...,0.245191,0.000000,0.000000,0.000000,None,None,None,None,None,1986-12-31
155,AAPL,1986-06-30,1986,Q3,USD,0.096356,0.096356,0.096356,0.096356,0.015723,...,0.000000,0.000000,0.000000,0.000000,None,None,None,None,None,1986-09-30
156,AAPL,1986-03-31,1986,Q2,USD,-0.234126,-0.234126,-0.234126,-0.234126,-0.441125,...,0.000000,0.000000,0.000000,0.000000,None,None,None,None,None,1986-06-30
157,AAPL,1985-12-31,1986,Q1,USD,0.303149,0.303149,1.392199,1.392199,1.551570,...,0.000000,0.000000,0.000000,0.000000,None,None,None,None,None,1986-03-31


## self calculation financial ratio

In [ ]:
# === Code Check for Financial Ratio Calculation ===

import pandas as pd
import json
import numpy as np

def calculate_available_ratios(financial_data):
    """
    Calculate financial ratios using ONLY fields that are actually available in the data.
    Strictly uses only existing fields without inventing or making up any data.
    """
    data = financial_data

    def safe_divide(numerator, denominator):
        try:
            if denominator == 0 or denominator is None or pd.isna(denominator):
                return np.nan
            return numerator / denominator
        except Exception as e:
            print(f"Error in safe_divide: {e}")
            return np.nan

    def field_exists(field_name):
        return field_name in data and data[field_name] is not None and not pd.isna(data[field_name])

    ratios = {
        'Date': data.get('date', ''),
        'Symbol': data.get('symbol', ''),
        'Period': data.get('period', ''),
        'Fiscal_Year': data.get('fiscalYear', ''),
    }

    # LIQUIDITY RATIOS
    if field_exists('totalCurrentAssets') and field_exists('totalCurrentLiabilities'):
        ratios['Current_Ratio'] = safe_divide(data['totalCurrentAssets'], data['totalCurrentLiabilities'])
    if field_exists('longTermDebt') and field_exists('totalAssets'):
        ratios['Debt_to_Assets'] = safe_divide(data['longTermDebt'], data['totalAssets'])
    if (field_exists('cashAndCashEquivalents') and field_exists('shortTermInvestments') and 
        field_exists('accountsReceivables') and field_exists('totalCurrentLiabilities')):
        quick_assets = data['cashAndCashEquivalents'] + data['shortTermInvestments'] + data['accountsReceivables']
        ratios['Quick_Ratio'] = safe_divide(quick_assets, data['totalCurrentLiabilities'])
    if field_exists('cashAndCashEquivalents') and field_exists('totalCurrentLiabilities'):
        ratios['Cash_Ratio'] = safe_divide(data['cashAndCashEquivalents'], data['totalCurrentLiabilities'])

    # PROFITABILITY RATIOS
    if field_exists('grossProfit') and field_exists('revenue'):
        ratios['Gross_Profit_Margin'] = safe_divide(data['grossProfit'], data['revenue'])
    if field_exists('operatingIncome') and field_exists('revenue'):
        ratios['Operating_Profit_Margin'] = safe_divide(data['operatingIncome'], data['revenue'])
    if field_exists('netIncome') and field_exists('revenue'):
        ratios['Net_Profit_Margin'] = safe_divide(data['netIncome'], data['revenue'])
    if field_exists('incomeBeforeTax') and field_exists('revenue'):
        ratios['Pre_Tax_Profit_Margin'] = safe_divide(data['incomeBeforeTax'], data['revenue'])
    if field_exists('netIncome') and field_exists('totalAssets'):
        ratios['Return_on_Assets_ROA'] = safe_divide(data['netIncome'], data['totalAssets'])
    if field_exists('netIncome') and field_exists('totalStockholdersEquity'):
        ratios['Return_on_Equity_ROE'] = safe_divide(data['netIncome'], data['totalStockholdersEquity'])

    # LEVERAGE/SOLVENCY RATIOS
    if field_exists('longTermDebt') and field_exists('totalStockholdersEquity'):
        ratios['Debt_to_Equity'] = safe_divide(data['longTermDebt'], data['totalStockholdersEquity'])
    if field_exists('longTermDebt') and field_exists('totalAssets'):
        ratios['Debt_to_Total_Assets'] = safe_divide(data['longTermDebt'], data['totalAssets'])
    if field_exists('totalDebt') and field_exists('totalAssets'):
        ratios['Debt_Ratio'] = safe_divide(data['totalDebt'], data['totalAssets'])
    if field_exists('totalAssets') and field_exists('totalStockholdersEquity'):
        ratios['Equity_Multiplier'] = safe_divide(data['totalAssets'], data['totalStockholdersEquity'])
    if field_exists('totalDebt') and field_exists('cashAndCashEquivalents') and field_exists('ebitda'):
        net_debt = data['totalDebt'] - data['cashAndCashEquivalents']
        ratios['Net_Debt_to_EBITDA'] = safe_divide(net_debt, data['ebitda'])
    if field_exists('ebit') and field_exists('interestExpense') and data['interestExpense'] != 0:
        ratios['Interest_Coverage_Ratio'] = safe_divide(data['ebit'], data['interestExpense'])

    # EFFICIENCY RATIOS
    if field_exists('revenue') and field_exists('totalAssets'):
        ratios['Asset_Turnover'] = safe_divide(data['revenue'], data['totalAssets'])
    if field_exists('revenue') and field_exists('accountsReceivables') and data['accountsReceivables'] != 0:
        ratios['Receivables_Turnover'] = safe_divide(data['revenue'], data['accountsReceivables'])
    if field_exists('revenue') and field_exists('propertyPlantEquipmentNet') and data['propertyPlantEquipmentNet'] != 0:
        ratios['Fixed_Asset_Turnover'] = safe_divide(data['revenue'], data['propertyPlantEquipmentNet'])

    # CASH FLOW RATIOS
    if field_exists('operatingCashFlow') and field_exists('revenue'):
        ratios['Operating_Cash_Flow_to_Revenue'] = safe_divide(data['operatingCashFlow'], data['revenue'])
    if field_exists('freeCashFlow') and field_exists('revenue'):
        ratios['Free_Cash_Flow_to_Revenue'] = safe_divide(data['freeCashFlow'], data['revenue'])
    if field_exists('freeCashFlow') and field_exists('operatingCashFlow') and data['operatingCashFlow'] != 0:
        ratios['Free_Cash_Flow_to_Operating_Cash_Flow'] = safe_divide(data['freeCashFlow'], data['operatingCashFlow'])
    if field_exists('capitalExpenditure') and data['capitalExpenditure'] != 0 and field_exists('operatingCashFlow'):
        capex = abs(data['capitalExpenditure'])
        ratios['capital_expenditure_coverage_ratio'] = safe_divide(data['operatingCashFlow'], capex)

    # PER SHARE METRICS
    if field_exists('eps'):
        ratios['Earnings_Per_Share'] = data['eps']
    if field_exists('revenue') and field_exists('weightedAverageShsOut'):
        ratios['Revenue_Per_Share'] = safe_divide(data['revenue'], data['weightedAverageShsOut'])
    if field_exists('cashAndCashEquivalents') and field_exists('weightedAverageShsOut'):
        ratios['Cash_Per_Share'] = safe_divide(data['cashAndCashEquivalents'], data['weightedAverageShsOut'])
    if field_exists('operatingCashFlow') and field_exists('weightedAverageShsOut'):
        ratios['Operating_Cash_Flow_Per_Share'] = safe_divide(data['operatingCashFlow'], data['weightedAverageShsOut'])
    if field_exists('freeCashFlow') and field_exists('weightedAverageShsOut'):
        ratios['Free_Cash_Flow_Per_Share'] = safe_divide(data['freeCashFlow'], data['weightedAverageShsOut'])

    # TAX RATIOS
    if field_exists('incomeTaxExpense') and field_exists('incomeBeforeTax') and data['incomeBeforeTax'] != 0:
        ratios['Effective_Tax_Rate'] = safe_divide(data['incomeTaxExpense'], data['incomeBeforeTax'])

    # WORKING CAPITAL
    if field_exists('totalCurrentAssets') and field_exists('totalCurrentLiabilities'):
        ratios['Working_Capital'] = data['totalCurrentAssets'] - data['totalCurrentLiabilities']

    # OTHER RATIOS
    if field_exists('commonDividendsPaid') and field_exists('netIncome') and data['netIncome'] != 0:
        dividends_paid = abs(data['commonDividendsPaid'])
        ratios['Dividend_Payout_Ratio'] = safe_divide(dividends_paid, data['netIncome'])
    if (field_exists('totalAssets') and field_exists('intangibleAssets') and 
        field_exists('weightedAverageShsOut')):
        tangible_assets = data['totalAssets'] - data['intangibleAssets']
        ratios['Tangible_Book_Value_Per_Share'] = safe_divide(tangible_assets, data['weightedAverageShsOut'])

    return ratios

def process_financial_data_file(filename):
    """
    Process financial data from JSON file and calculate all available ratios
    """
    try:
        with open(filename, 'r') as file:
            financial_data = json.load(file)
        if isinstance(financial_data, list):
            all_ratios = []
            for record in financial_data:
                ratios = calculate_available_ratios(record)
                all_ratios.append(ratios)
            df = pd.DataFrame(all_ratios)
            return df
        else:
            ratios = calculate_available_ratios(financial_data)
            return pd.DataFrame([ratios])
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found.")
        return None
    except json.JSONDecodeError:
        print(f"Error: Invalid JSON format in file '{filename}'.")
        return None
    except Exception as e:
        print(f"Error processing file: {str(e)}")
        return None

# === Simple Check ===
if __name__ == "__main__":
    filename = "sample-data/AAPL_fundamental_data.json"
    print("Checking financial ratio calculation code...")
    ratios_df = process_financial_data_file(filename)
    if ratios_df is not None:
        print(f"Number of periods: {len(ratios_df)}")
        print(f"Number of ratio columns: {len(ratios_df.columns)}")
        print("First few rows:")
        print(ratios_df.head())
        print("\nColumns:")
        print(list(ratios_df.columns))
        # Check for NaNs in important ratios
        important = ['Current_Ratio', 'Net_Profit_Margin', 'Return_on_Equity_ROE']
        for col in important:
            if col in ratios_df.columns:
                print(f"{col} NaN count: {ratios_df[col].isna().sum()}")
    else:
        print("Failed to process the file. Please check the file path and format.")


Checking financial ratio calculation code...
Number of periods: 159
Number of ratio columns: 36
First few rows:
         Date Symbol Period Fiscal_Year  Current_Ratio  Debt_to_Assets  \
0  2025-03-29   AAPL     Q2        2025       0.820870        0.237193   
1  2024-12-28   AAPL     Q1        2025       0.922938        0.243998   
2  2024-09-28   AAPL     Q4        2024       0.867313        0.234944   
3  2024-06-29   AAPL     Q3        2024       0.952980        0.259930   
4  2024-03-30   AAPL     Q2        2024       1.037102        0.272164   

   Quick_Ratio  Cash_Ratio  Gross_Profit_Margin  Operating_Profit_Margin  ...  \
0     0.402377    0.194797             0.470506                 0.310291  ...   
1     0.419340    0.209878             0.468825                 0.344586  ...   
2     0.239410    0.169753             0.462225                 0.311714  ...   
3     0.453618    0.194227             0.462572                 0.295557  ...   
4     0.613114    0.264048            

In [16]:
len(ratios_df.columns)


36